# FireSight-IR | Module 1b — ERA5 Atmospheric Profile Co-location

**Project:** FireSight-IR — FireSat Protoflight-aligned wildfire detection pipeline  
**Author:** Emmanuel Ibekwe | github.com/Ibekwemmanuel7  
**Module:** 1b of 9 — Atmospheric context: ERA5 temperature, humidity, PBL height + MODIS AOD  
**Platform:** Google Colab  
**Depends on:** Module 1a outputs in Google Drive

---

## Motivation — why atmospheric state matters

The FireSat Protoflight first-light imagery released by Earth Fire Alliance in July 2025 demonstrated
exactly the intelligence challenge FireSight-IR is built to address. A small roadside fire near Medford,
Oregon — undetected by all existing satellites — was captured by FireSat's MWIR channel. The Ontario
Nipigon 6 Fire imagery showed the LWIR channel picking up a 2020 burn scar being warmed by the sun,
producing a thermal signature that could be confused with active fire. These are not edge cases —
they are the operational reality of infrared fire detection under variable atmospheric and surface conditions.

Most ML fire detection models treat the atmosphere as noise to be tolerated. FireSight-IR treats it
as a physical constraint to be encoded. This module builds that constraint layer.

## What this notebook does

1. Authenticates with the Copernicus Climate Data Store (CDS) for ERA5 access
2. For each unique fire day in the FIRMS dataset, downloads ERA5:
   - Near-surface air temperature (2m)
   - Boundary layer height (PBL)
   - Total column water vapour
   - Surface pressure
   - Temperature at pressure levels (850, 700, 500 hPa — a simplified atmospheric profile)
3. Spatially interpolates ERA5 fields to each VIIRS fire pixel location
4. Downloads MODIS MOD04 Aerosol Optical Depth (AOD) via earthaccess
5. Co-locates AOD with fire pixels
6. Merges atmospheric context with the VIIRS fire pixel parquet files from Module 1a
7. Saves enriched parquet files to Google Drive

## Physical significance of each variable

| Variable | Why it matters for FireSight-IR |
|---|---|
| 2m temperature | Surface thermal baseline — separates fire signal from hot ambient conditions |
| PBL height | Controls smoke ventilation and fire behaviour — high PBL = active convection |
| Column water vapour | Attenuates MWIR/LWIR signal via Beer-Lambert transmittance |
| Surface pressure | Required for Beer-Lambert atmospheric path calculation |
| Temperature profile | Atmospheric stability index — unstable atmosphere = explosive fire growth |
| MODIS AOD | Smoke and aerosol loading — directly affects IR transmittance to satellite |

## Output

```
Google Drive: firesight-ir/
  data/
    processed/
      viirs_fp_atm/     ← enriched parquet files (VIIRS + ERA5 + AOD)
    raw/
      era5/             ← raw ERA5 NetCDF files (cached)
```

---
## Section 0 — Mount Drive and install packages

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
print('Drive mounted.')

In [ ]:
!pip install cdsapi earthaccess xarray netCDF4 scipy pandas numpy pyarrow tqdm -q

import cdsapi
import earthaccess
import xarray as xr
import pandas as pd
import numpy as np
import scipy.interpolate
import os, time, warnings
from pathlib import Path
from tqdm.notebook import tqdm

# cdsapi does not expose __version__ — check it loaded instead
import importlib.metadata
try:
    cdsapi_ver = importlib.metadata.version("cdsapi")
except Exception:
    cdsapi_ver = "installed (version unknown)"

print(f"cdsapi    : {cdsapi_ver}")
print(f"xarray    : {xr.__version__}")
print(f"pandas    : {pd.__version__}")
print("All imports OK")


---
## Section 1 — Configuration

In [ ]:
# ── Paths ─────────────────────────────────────────────────────────────────────
BASE_DIR     = '/content/drive/MyDrive/firesight-ir'
FIRMS_DIR    = f'{BASE_DIR}/data/raw/firms'
VIIRS_DIR    = f'{BASE_DIR}/data/processed/viirs_fp'
ERA5_DIR     = f'{BASE_DIR}/data/raw/era5'
ATM_DIR      = f'{BASE_DIR}/data/processed/viirs_fp_atm'
FIGURES_DIR  = f'{BASE_DIR}/figures'

for d in [ERA5_DIR, ATM_DIR, FIGURES_DIR]:
    os.makedirs(d, exist_ok=True)

# ── Domain ────────────────────────────────────────────────────────────────────
# ERA5 area format: [North, West, South, East]
ERA5_AREA    = [49, -125, 32, -109]
BBOX_TUPLE   = (-125, 32, -109, 49)   # for earthaccess

# ── Years ─────────────────────────────────────────────────────────────────────
ALL_YEARS         = [2018, 2019, 2020, 2021, 2022, 2023]
FIRE_SEASON_MONTHS = [5, 6, 7, 8, 9, 10]

# ── ERA5 pressure levels for atmospheric profile ──────────────────────────────
# 1000=surface, 850=~1.5km, 700=~3km, 500=~5.5km, 300=~9km
PRESSURE_LEVELS  = ['1000', '850', '700', '500', '300']

print('Configuration set.')
print(f'  ERA5 cache   : {ERA5_DIR}')
print(f'  Output       : {ATM_DIR}')

---
## Section 2 — CDS Authentication

The Copernicus CDS API requires a `.cdsapirc` file with your credentials.
We write it programmatically — you only need to do this once per Colab session.

> **Security note:** After your first successful run, regenerate your API key at
> https://cds.climate.copernicus.eu/profile to prevent unauthorised use.

In [ ]:
# ── Write CDS credentials ─────────────────────────────────────────────────────
# Replace with your actual key if you have regenerated it
CDS_URL = 'https://cds.climate.copernicus.eu/api'
CDS_KEY = '202e0286-d5ac-4f0c-8cfb-367a45a0746c'  # regenerate after first use

cdsapirc = f'url: {CDS_URL}\nkey: {CDS_KEY}\n'
rc_path  = Path.home() / '.cdsapirc'
rc_path.write_text(cdsapirc)
print(f'CDS credentials written to {rc_path}')

# Test connection
try:
    client = cdsapi.Client(quiet=True)
    print('CDS authentication: OK')
except Exception as e:
    print(f'CDS authentication failed: {e}')
    print('Check your API key at https://cds.climate.copernicus.eu/profile')

---
## Section 3 — Load Module 1a data

Load the VIIRS fire pixel DataFrames produced in Module 1a.
These are the records we need to enrich with atmospheric context.

In [ ]:
# ── Load all VIIRS fire pixel files ───────────────────────────────────────────
viirs_data = {}
for year in ALL_YEARS:
    fp = f'{VIIRS_DIR}/viirs_fp_{year}.parquet'
    if os.path.exists(fp):
        df = pd.read_parquet(fp)
        df['date'] = pd.to_datetime(df['date'])
        viirs_data[year] = df
        print(f'  {year}: {len(df):>9,} fire pixels loaded')
    else:
        print(f'  {year}: NOT FOUND — run Module 1a first')

viirs_all = pd.concat(viirs_data.values(), ignore_index=True)
print(f'\nTotal: {len(viirs_all):,} fire pixels')
print(f'Date range: {viirs_all["date"].min().date()} → {viirs_all["date"].max().date()}')

# Get unique fire dates
fire_dates = sorted(viirs_all['date'].dt.date.unique())
print(f'Unique fire dates: {len(fire_dates)}')

---
## Section 4 — ERA5 download functions

### ERA5 variables we request

**Single-level (surface) variables:**
- `2m_temperature` — near-surface air temperature (K)
- `boundary_layer_height` — planetary boundary layer height (m)
- `total_column_water_vapour` — column-integrated water vapour (kg/m²)
- `surface_pressure` — surface pressure (Pa)

**Pressure-level variables** (at 1000, 850, 700, 500, 300 hPa):
- `temperature` — atmospheric temperature profile (K)
- `specific_humidity` — moisture profile (kg/kg)

### Download strategy
We download one ERA5 file per month (not per day) to minimise API calls.
ERA5 has hourly data — we select the two hours closest to typical VIIRS overpass
times (0100 UTC and 2200 UTC) to keep file sizes manageable.

In [ ]:
def download_era5_single_level(year, month, out_dir, client):
    """
    Download ERA5 single-level variables for one month.
    Returns path to downloaded NetCDF file.
    """
    out_path = f'{out_dir}/era5_surface_{year}{month:02d}.nc'
    if os.path.exists(out_path):
        return out_path

    print(f'  Downloading ERA5 surface {year}-{month:02d}...')
    try:
        client.retrieve(
            'reanalysis-era5-single-levels',
            {
                'product_type': 'reanalysis',
                'variable': [
                    '2m_temperature',
                    'boundary_layer_height',
                    'total_column_water_vapour',
                    'surface_pressure',
                ],
                'year'  : str(year),
                'month' : f'{month:02d}',
                'day'   : [f'{d:02d}' for d in range(1, 32)],
                # VIIRS overpass times: ~0130 UTC (day) and ~2130 UTC (night)
                'time'  : ['01:00', '13:00', '22:00'],
                'area'  : ERA5_AREA,
                'format': 'netcdf',
            },
            out_path
        )
        print(f'    Saved → {os.path.basename(out_path)}')
        return out_path
    except Exception as e:
        print(f'    ERROR: {e}')
        return None


def download_era5_pressure_levels(year, month, out_dir, client):
    """
    Download ERA5 pressure-level temperature and humidity for one month.
    """
    out_path = f'{out_dir}/era5_pressure_{year}{month:02d}.nc'
    if os.path.exists(out_path):
        return out_path

    print(f'  Downloading ERA5 pressure levels {year}-{month:02d}...')
    try:
        client.retrieve(
            'reanalysis-era5-pressure-levels',
            {
                'product_type'   : 'reanalysis',
                'variable'       : ['temperature', 'specific_humidity'],
                'pressure_level' : PRESSURE_LEVELS,
                'year'           : str(year),
                'month'          : f'{month:02d}',
                'day'            : [f'{d:02d}' for d in range(1, 32)],
                'time'           : ['01:00', '13:00', '22:00'],
                'area'           : ERA5_AREA,
                'format'         : 'netcdf',
            },
            out_path
        )
        print(f'    Saved → {os.path.basename(out_path)}')
        return out_path
    except Exception as e:
        print(f'    ERROR: {e}')
        return None


print('ERA5 download functions defined.')

---
## Section 5 — Co-location functions

For each fire pixel, we find the ERA5 grid cell nearest in space and time
and extract the atmospheric state variables at that location.

ERA5 native resolution is ~31 km. Since fire pixels are at 375 m, we do
bilinear spatial interpolation to the exact fire pixel coordinates.

In [ ]:
def find_nearest_time(ds, target_time):
    """
    Find the index of the ERA5 time step nearest to target_time.
    ERA5 times are stored as numpy datetime64.
    """
    times = pd.to_datetime(ds.time.values)
    deltas = abs(times - pd.Timestamp(target_time))
    return int(deltas.argmin())


def interpolate_era5_to_points(ds, var_name, lats, lons, time_idx):
    """
    Bilinear interpolation of an ERA5 variable to a set of (lat, lon) points.

    Parameters
    ----------
    ds       : xr.Dataset — ERA5 dataset
    var_name : str        — variable name in dataset
    lats     : array      — target latitudes
    lons     : array      — target longitudes
    time_idx : int        — time step index

    Returns
    -------
    np.array of interpolated values
    """
    da = ds[var_name].isel(valid_time=time_idx)

    # Handle ERA5 coordinate naming (latitude/longitude or lat/lon)
    lat_name = 'latitude'  if 'latitude'  in ds.coords else 'lat'
    lon_name = 'longitude' if 'longitude' in ds.coords else 'lon'

    grid_lats = ds[lat_name].values
    grid_lons = ds[lon_name].values

    # ERA5 lats are descending (90→-90) — flip for interpolation
    if grid_lats[0] > grid_lats[-1]:
        da        = da.isel({lat_name: slice(None, None, -1)})
        grid_lats = grid_lats[::-1]

    values = da.values

    # Bilinear interpolation
    interp_fn = scipy.interpolate.RegularGridInterpolator(
        (grid_lats, grid_lons),
        values,
        method='linear',
        bounds_error=False,
        fill_value=np.nan
    )
    return interp_fn(np.column_stack([lats, lons]))


def extract_era5_for_day(df_day, surface_path, pressure_path):
    """
    Extract ERA5 atmospheric context for all fire pixels on one day.

    Parameters
    ----------
    df_day        : pd.DataFrame — fire pixels for one day (from Module 1a)
    surface_path  : str          — path to ERA5 single-level NetCDF
    pressure_path : str          — path to ERA5 pressure-level NetCDF

    Returns
    -------
    pd.DataFrame with atmospheric columns added
    """
    lats = df_day['latitude'].values
    lons = df_day['longitude'].values

    # Use midday (13:00 UTC) as representative time — captures daytime fire activity
    target_dt = pd.Timestamp(df_day['date'].iloc[0]).replace(hour=13)

    result = df_day.copy()

    # ── Surface variables ─────────────────────────────────────────────────────
    if surface_path and os.path.exists(surface_path):
        with warnings.catch_warnings():
            warnings.simplefilter('ignore')
            ds_sfc = xr.open_dataset(surface_path)

        t_idx = find_nearest_time(ds_sfc, target_dt)

        var_map = {
            't2m'  : 'era5_t2m',       # 2m temperature (K)
            'blh'  : 'era5_pbl',        # boundary layer height (m)
            'tcwv' : 'era5_tcwv',       # total column water vapour (kg/m²)
            'sp'   : 'era5_sp',         # surface pressure (Pa)
        }
        for era5_var, col_name in var_map.items():
            if era5_var in ds_sfc.data_vars:
                result[col_name] = interpolate_era5_to_points(
                    ds_sfc, era5_var, lats, lons, t_idx
                )
        ds_sfc.close()

    # ── Pressure level variables ──────────────────────────────────────────────
    if pressure_path and os.path.exists(pressure_path):
        with warnings.catch_warnings():
            warnings.simplefilter('ignore')
            ds_pl = xr.open_dataset(pressure_path)

        t_idx = find_nearest_time(ds_pl, target_dt)

        # Extract temperature at each pressure level
        for plev in PRESSURE_LEVELS:
            plev_int = int(plev)
            if 'pressure_level' in ds_pl.dims:
                try:
                    da_t = ds_pl['t'].sel(pressure_level=plev_int)
                    da_q = ds_pl['q'].sel(pressure_level=plev_int)

                    # Bilinear interpolation at this level
                    lat_name = 'latitude'  if 'latitude'  in ds_pl.coords else 'lat'
                    lon_name = 'longitude' if 'longitude' in ds_pl.coords else 'lon'
                    grid_lats = ds_pl[lat_name].values
                    grid_lons = ds_pl[lon_name].values

                    if grid_lats[0] > grid_lats[-1]:
                        t_vals = da_t.isel(valid_time=t_idx).values[::-1, :]
                        q_vals = da_q.isel(valid_time=t_idx).values[::-1, :]
                        grid_lats = grid_lats[::-1]
                    else:
                        t_vals = da_t.isel(valid_time=t_idx).values
                        q_vals = da_q.isel(valid_time=t_idx).values

                    fn_t = scipy.interpolate.RegularGridInterpolator(
                        (grid_lats, grid_lons), t_vals,
                        method='linear', bounds_error=False, fill_value=np.nan)
                    fn_q = scipy.interpolate.RegularGridInterpolator(
                        (grid_lats, grid_lons), q_vals,
                        method='linear', bounds_error=False, fill_value=np.nan)

                    pts = np.column_stack([lats, lons])
                    result[f'era5_t_{plev}hPa'] = fn_t(pts)
                    result[f'era5_q_{plev}hPa'] = fn_q(pts)
                except Exception:
                    pass

        ds_pl.close()

    # ── Derived atmospheric features ──────────────────────────────────────────
    # Beer-Lambert transmittance proxy: τ = exp(-k * TCWV)
    # k = 0.05 is an approximation for MWIR 3.7µm channel
    if 'era5_tcwv' in result.columns:
        result['beer_lambert_proxy'] = np.exp(-0.05 * result['era5_tcwv'])

    # Atmospheric instability index: T_850 - T_500 (crude lapse rate)
    if 'era5_t_850hPa' in result.columns and 'era5_t_500hPa' in result.columns:
        result['atm_instability'] = result['era5_t_850hPa'] - result['era5_t_500hPa']

    return result


print('Co-location functions defined.')

---
## Section 6 — Test on one month before full run

Download ERA5 for September 2020 (peak fire month) and verify co-location
works correctly before committing to all 6 years.

In [ ]:
# ── Download ERA5 for September 2020 ─────────────────────────────────────────
client = cdsapi.Client(quiet=True)

print('Downloading ERA5 test data (Sep 2020)...')
sfc_path_test = download_era5_single_level(2020, 9, ERA5_DIR, client)
pl_path_test  = download_era5_pressure_levels(2020, 9, ERA5_DIR, client)

print(f'\nSurface file : {sfc_path_test}')
print(f'Pressure file: {pl_path_test}')

In [ ]:
# ── Inspect ERA5 file structure ───────────────────────────────────────────────
if sfc_path_test:
    ds = xr.open_dataset(sfc_path_test)
    print('=== ERA5 Surface Dataset ===')
    print(ds)
    print('\nVariables:', list(ds.data_vars))
    print('Time steps:', len(ds.valid_time), '| First:', pd.Timestamp(ds.valid_time.values[0]))
    ds.close()

In [ ]:
# ── Test co-location on one day ───────────────────────────────────────────────
# Use September 8 2020 — peak Creek Fire day
test_date = pd.Timestamp('2020-09-08')
df_test_day = viirs_all[viirs_all['date'].dt.date == test_date.date()].copy()
print(f'Fire pixels on {test_date.date()}: {len(df_test_day):,}')

# Run co-location
df_enriched_test = extract_era5_for_day(
    df_test_day, sfc_path_test, pl_path_test
)

# Check results
print('\n=== Co-location test results ===')
atm_cols = [c for c in df_enriched_test.columns if 'era5' in c or 'beer' in c or 'atm_' in c]
print(f'Atmospheric columns added: {atm_cols}')
print('\nSample statistics:')
print(df_enriched_test[atm_cols].describe().round(3))

In [ ]:
# ── Visualise co-location: PBL height overlaid on fire detections ─────────────
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors

fig, axes = plt.subplots(1, 2, figsize=(14, 6), facecolor='#0a0a0a')
fig.patch.set_facecolor('#0a0a0a')

# Left: fire FRP
ax1 = axes[0]
ax1.set_facecolor('#0a0a0a')
sc1 = ax1.scatter(
    df_enriched_test['longitude'], df_enriched_test['latitude'],
    c=df_enriched_test['frp_mw'], cmap='hot',
    s=1, alpha=0.8, vmin=0, vmax=500
)
ax1.set_title('Fire Radiative Power (MW)\n2020-09-08', color='white', fontsize=11)
ax1.tick_params(colors='#666666')
plt.colorbar(sc1, ax=ax1, label='FRP (MW)').ax.yaxis.set_tick_params(labelcolor='#888888')

# Right: PBL height
ax2 = axes[1]
ax2.set_facecolor('#0a0a0a')
sc2 = ax2.scatter(
    df_enriched_test['longitude'], df_enriched_test['latitude'],
    c=df_enriched_test['era5_pbl'], cmap='YlOrRd',
    s=1, alpha=0.8
)
ax2.set_title('ERA5 PBL Height (m)\n2020-09-08', color='white', fontsize=11)
ax2.tick_params(colors='#666666')
plt.colorbar(sc2, ax=ax2, label='PBL Height (m)').ax.yaxis.set_tick_params(labelcolor='#888888')

for ax in axes:
    ax.set_xlim(-125, -109)
    ax.set_ylim(32, 49)
    for spine in ax.spines.values():
        spine.set_edgecolor('#333333')

fig.suptitle('FireSight-IR | ERA5 Co-location Verification — Peak Creek Fire Day',
             color='white', fontsize=13, y=1.01)
plt.tight_layout()
plt.savefig(f'{FIGURES_DIR}/era5_colocation_test.png',
            dpi=150, bbox_inches='tight', facecolor='#0a0a0a')
plt.show()
print('Co-location test figure saved.')

---
## Section 7 — Full multi-year ERA5 download and co-location

Downloads ERA5 for all fire-season months across all years, then
co-locates atmospheric variables with all VIIRS fire pixels.

**Expected download size:** ~200–400 MB per month × 36 months = ~10–15 GB total  
**Expected runtime:** 2–4 hours (dominated by CDS queue wait times)

CDS requests are queued on the server side — sometimes instantly, sometimes
5–10 minutes per request. This is normal. The `cdsapi` client waits automatically.

In [ ]:
# ── Download all ERA5 months ──────────────────────────────────────────────────
# Only downloads months that have fire activity in our FIRMS data
# and haven't been downloaded yet

# Get unique year-month combinations from fire data
fire_months = (
    viirs_all[['date']]
    .assign(year=viirs_all['date'].dt.year,
            month=viirs_all['date'].dt.month)
    .drop_duplicates(['year', 'month'])
    .sort_values(['year', 'month'])
)

print(f'Year-month combinations to download: {len(fire_months)}')
print(fire_months[['year', 'month']].to_string(index=False))

In [ ]:
# ── Download ERA5 (run this — takes 2-4 hours total) ─────────────────────────
client = cdsapi.Client(quiet=False)  # quiet=False shows download progress

sfc_paths = {}
pl_paths  = {}

for _, row in fire_months.iterrows():
    year, month = int(row['year']), int(row['month'])
    print(f'\n── {year}-{month:02d} ──')

    sfc = download_era5_single_level(year, month, ERA5_DIR, client)
    pl  = download_era5_pressure_levels(year, month, ERA5_DIR, client)

    if sfc: sfc_paths[(year, month)] = sfc
    if pl:  pl_paths[(year, month)]  = pl

print(f'\nERA5 surface files   : {len(sfc_paths)}')
print(f'ERA5 pressure files  : {len(pl_paths)}')

In [ ]:
# ── Co-locate ERA5 with all fire pixels ───────────────────────────────────────
def process_year_era5(year, viirs_data, sfc_paths, pl_paths, out_dir):
    """
    Co-locate ERA5 atmospheric variables with all fire pixels for one year.
    Saves enriched parquet to out_dir.
    """
    out_path = f'{out_dir}/viirs_fp_atm_{year}.parquet'
    if os.path.exists(out_path):
        print(f'  {year} already processed — skipping')
        return pd.read_parquet(out_path)

    df_year = viirs_data[year].copy()
    df_year['date'] = pd.to_datetime(df_year['date'])

    enriched_days = []
    unique_dates  = sorted(df_year['date'].dt.date.unique())
    print(f'  {year}: co-locating {len(unique_dates)} days...')

    for d in unique_dates:
        month    = d.month
        df_day   = df_year[df_year['date'].dt.date == d]
        sfc_path = sfc_paths.get((year, month))
        pl_path  = pl_paths.get((year, month))

        if sfc_path is None and pl_path is None:
            enriched_days.append(df_day)
            continue

        try:
            enriched = extract_era5_for_day(df_day, sfc_path, pl_path)
            enriched_days.append(enriched)
        except Exception as e:
            print(f'    Warning: {d} failed: {e}')
            enriched_days.append(df_day)

    df_out = pd.concat(enriched_days, ignore_index=True)
    df_out.to_parquet(out_path, index=False)
    print(f'  {year}: {len(df_out):,} enriched pixels saved → Drive')
    return df_out


# ── Run all years ─────────────────────────────────────────────────────────────
enriched_data = {}
for year in ALL_YEARS:
    if year not in viirs_data:
        continue
    print(f'\n{"="*45}')
    print(f'  Co-locating ERA5 for {year}')
    print(f'{"="*45}')
    df = process_year_era5(year, viirs_data, sfc_paths, pl_paths, ATM_DIR)
    if df is not None:
        enriched_data[year] = df

print('\nERA5 co-location complete.')

---
## Section 8 — MODIS AOD co-location

MODIS MOD04 provides Aerosol Optical Depth (AOD) at 1–10 km resolution daily.
AOD is the key input to the Beer-Lambert atmospheric transmittance term
in the physics-informed loss function:

```
τ(λ) = exp(-AOD / cos(θ_view))
```

where θ_view is the VIIRS view zenith angle already in the fire pixel DataFrame.

In [ ]:
# ── Authenticate earthaccess for MODIS AOD ────────────────────────────────────
auth = earthaccess.login(strategy='interactive')
print(f'Earthdata authenticated: {auth.authenticated}')

In [ ]:
def fetch_modis_aod_day(date_str, bbox_tuple):
    """
    Search and open MODIS MOD04_L2 AOD granules for one day.
    Returns a DataFrame with (latitude, longitude, aod_550nm) columns,
    or None if no granules found.

    MOD04_L2 is the MODIS Terra 10 km aerosol product.
    We use the 'Optical_Depth_Land_And_Ocean' variable at 550 nm.
    """
    try:
        results = earthaccess.search_data(
            short_name   = 'MOD04_L2',
            bounding_box = bbox_tuple,
            temporal     = (date_str, date_str),
        )
        if not results:
            return None

        files   = earthaccess.open(results)
        aod_dfs = []

        for f in files:
            try:
                with warnings.catch_warnings():
                    warnings.simplefilter('ignore')
                    ds = xr.open_dataset(f, group='mod04', engine='h5netcdf')

                if 'Optical_Depth_Land_And_Ocean' not in ds.data_vars:
                    ds.close()
                    continue

                aod  = ds['Optical_Depth_Land_And_Ocean'].values.flatten()
                lats = ds['Latitude'].values.flatten()
                lons = ds['Longitude'].values.flatten()

                # Apply scale factor and filter fill values
                scale  = ds['Optical_Depth_Land_And_Ocean'].attrs.get('scale_factor', 0.001)
                aod    = aod.astype(float) * scale
                valid  = (lats >= -90) & (lats <= 90) & (aod > -0.05) & (aod < 5.0)

                aod_dfs.append(pd.DataFrame({
                    'lat_aod': lats[valid],
                    'lon_aod': lons[valid],
                    'aod_550': aod[valid],
                }))
                ds.close()
            except Exception:
                continue

        if not aod_dfs:
            return None
        return pd.concat(aod_dfs, ignore_index=True)

    except Exception as e:
        return None


def add_aod_to_fire_pixels(df_fire, df_aod):
    """
    Nearest-neighbour match AOD values to fire pixel locations.
    Uses scipy KDTree for fast spatial lookup.
    """
    from scipy.spatial import cKDTree

    if df_aod is None or len(df_aod) == 0:
        df_fire['aod_550'] = np.nan
        return df_fire

    # Build KD-tree on AOD grid points
    tree = cKDTree(df_aod[['lat_aod', 'lon_aod']].values)

    # Query nearest AOD point for each fire pixel
    dists, idxs = tree.query(
        df_fire[['latitude', 'longitude']].values,
        k=1, workers=-1
    )

    # Only assign AOD if nearest point is within 0.15 degrees (~15 km)
    aod_vals = df_aod['aod_550'].values[idxs]
    aod_vals[dists > 0.15] = np.nan

    df_fire = df_fire.copy()
    df_fire['aod_550'] = aod_vals

    # Compute Beer-Lambert transmittance using AOD and view zenith angle
    if 'view_zen' in df_fire.columns:
        cos_vza = np.cos(np.radians(df_fire['view_zen'].values))
        cos_vza = np.clip(cos_vza, 0.1, 1.0)  # avoid division by zero
        df_fire['transmittance_aod'] = np.exp(
            -df_fire['aod_550'].fillna(0.2) / cos_vza
        )

    return df_fire


print('MODIS AOD functions defined.')

In [ ]:
# ── Test AOD co-location on peak fire day ─────────────────────────────────────
print('Testing MODIS AOD co-location on 2020-09-08...')
df_aod_test = fetch_modis_aod_day('2020-09-08', BBOX_TUPLE)

if df_aod_test is not None:
    print(f'  AOD points found: {len(df_aod_test):,}')
    print(f'  AOD range: {df_aod_test["aod_550"].min():.3f} – {df_aod_test["aod_550"].max():.3f}')

    # Add to test enriched data
    df_with_aod = add_aod_to_fire_pixels(df_enriched_test, df_aod_test)
    aod_coverage = df_with_aod['aod_550'].notna().mean() * 100
    print(f'  AOD coverage of fire pixels: {aod_coverage:.1f}%')
    print(f'  Mean AOD at fire pixels: {df_with_aod["aod_550"].mean():.3f}')
    print(f'  Mean transmittance: {df_with_aod["transmittance_aod"].mean():.3f}')
else:
    print('  No MODIS AOD granules found for this date — will use fill value 0.2')

---
## Section 9 — Final verification and summary

Verify the enriched dataset is complete and physically reasonable
before passing to Module 1c (harmonization and patch extraction).

In [ ]:
# ── Load and verify enriched files ────────────────────────────────────────────
enriched_files = sorted([
    f for f in os.listdir(ATM_DIR)
    if f.startswith('viirs_fp_atm_') and f.endswith('.parquet')
])

print('=== Module 1b Summary ===')
print(f'Enriched files in Drive: {len(enriched_files)}')
print()

total_pixels = 0
for fname in enriched_files:
    df = pd.read_parquet(f'{ATM_DIR}/{fname}')
    atm_cols   = [c for c in df.columns if 'era5' in c or 'aod' in c or
                  'transmit' in c or 'beer' in c or 'atm_' in c]
    coverage   = df[atm_cols].notna().mean().mean() * 100 if atm_cols else 0
    total_pixels += len(df)
    year = fname.split('_')[-1].replace('.parquet', '')
    print(f'  {year}: {len(df):>9,} pixels | {len(atm_cols)} atm columns | '
          f'{coverage:.0f}% coverage')

print(f'\n  TOTAL: {total_pixels:,} enriched fire pixels')
print(f'\nFull column list (last file):')
print(df.columns.tolist())
print()
print('=== Module 1b Complete ===')
print('Next: Module 1c — harmonization, co-registration, patch extraction')

---
## Physics note: Beer-Lambert transmittance

The `transmittance_aod` column computed in this module is the first physics
constraint that will appear in the Module 3 loss function:

```
τ(λ) = exp(-AOD(λ) / cos(θ_view))
```

Where:
- `AOD` is the aerosol optical depth at 550 nm from MODIS MOD04
- `θ_view` is the VIIRS view zenith angle (already in fire pixel DataFrame)
- The division by `cos(θ_view)` accounts for longer atmospheric path at high
  view angles — a pixel at 60° zenith has twice the atmospheric path length
  of a nadir-looking pixel

This value directly enters the physics-informed loss function as:

```python
L_BeerLambert = MSE(predicted_transmittance, computed_transmittance_aod)
```

A model that assigns high fire probability to a pixel where the atmosphere
is optically thick (high AOD, low transmittance) without accounting for
the attenuation is physically inconsistent — and this loss term penalises
that inconsistency during training.

## Troubleshooting

| Issue | Cause | Fix |
|---|---|---|
| CDS authentication fails | Key expired or wrong | Regenerate at cds.climate.copernicus.eu/profile |
| CDS request queued for >30 min | Server load | Normal — client waits automatically |
| ERA5 variable names differ | Dataset version | Check `list(ds.data_vars)` and update var_map |
| AOD coverage < 50% | Cloud cover or no MODIS overpass | Expected — use fill value 0.2 for missing |
| `valid_time` not found | ERA5 coordinate naming | Try `time` instead of `valid_time` in xr.open_dataset |
| Beer-Lambert transmittance = 1.0 everywhere | AOD all NaN using fill 0 | Confirmed fill value 0.2 is set in `add_aod_to_fire_pixels` |